# Airbnb Dublin – Conversion Funnel Analysis

**Author:** Nemi Yossef Hai  
**LinkedIn:** [linkedin.com/in/nemi-yossef-hai](https://www.linkedin.com/in/nemi-yossef-hai)  
**Email:** nemiys@gmail.com

---

## Business Question

Airbnb's Dublin marketplace generates tens of thousands of user interactions every day — searches, contact requests, host replies, acceptances, and bookings. But a large portion of users who search never book.

This analysis asks: **Where do users drop off, and why?**

More specifically:
- What does the end-to-end conversion funnel look like, and at which stage is the biggest drop?
- Do behavioral signals (search filters, lead time) predict whether a user will convert?
- Which markets are most efficient, and which are leaving bookings on the table?
- Are there structural supply-side problems (idle listings) hurting conversion?

## Dataset Overview

Two tables extracted from the Airbnb Dublin marketplace:

| Table | Rows | Description |
|---|---|---|
| `searches` | ~130,000 | One row per search event. Includes user ID, search date, check-in/out dates, number of guests, and filter usage. |
| `contacts` | ~50,000 | One row per guest–host contact. Includes contact timestamp, reply timestamp, acceptance, and booking outcome. |

The two tables are linked via `id_user` (searches) / `id_guest` (contacts).

> **Note:** This dataset is real platform data from an Airbnb Dublin analytics exercise, pre-cleaned in Excel before Python analysis. The cleaning steps are documented in Part A.

---

## Structure

| Part | Topic |
|---|---|
| **A** | Data Cleaning, Feature Engineering & Funnel Methodology |
| **B** | Platform-Wide Findings |
| **C** | Market-by-Market Analysis |
| **Summary** | Business Recommendations |

In [ ]:
# ── Install / import ─────────────────────────────────────────────────────────
import urllib.request, os

BASE  = 'https://raw.githubusercontent.com/nemiys/airbnb-dublin-conversion-analysis-python/master/'
FILES = ['contacts_fixed.xlsx', 'searches_fixed.xlsx', 'Countries.csv']

for f in FILES:
    urllib.request.urlretrieve(BASE + f, f)
    print(f'Downloaded: {f}')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
RED  = '#FF5A5F'
TEAL = '#00A699'
DARK = '#484848'
print('Setup complete.')

---
# Part A – Data Preparation

## A1. Data Cleaning Log

Cleaning was performed in Excel before loading into Python. Key steps:

### Contacts table
- Converted `ts_contact_at`, `ts_reply_at`, `ts_accepted_at`, `ts_booking_at` from text to datetime.
- Replaced literal `"NULL"` strings with true blank cells to avoid type errors.
- Verified logical consistency: no rows exist where a booking occurred without a prior reply. All records passed.
- Clipped negative `lead_time_days` values (same-day check-ins) to 0.
- UUID columns preserved as text.
- **Derived columns already in file:** `is_replied`, `is_accepted`, `is_booked`, `reply_time_minutes`, `lead_time_days`, `length_of_stay`, `lead_time_category`.

### Searches table
- Missing values are true blanks (no NULL strings) — represent searches without specific dates.
- Date validation column `date_check` flags 25 logic errors and 11,849 date-less searches (valid).
- **Derived columns already in file:** `uses_price_filter`, `uses_room_filter`, `stay_category`, `n_searches`, `n_nights`, `search_lead_time`, `lead_time_category`.

## A2. Load & Verify Data

In [ ]:
contacts = pd.read_excel('contacts_fixed.xlsx')
searches = pd.read_excel('searches_fixed.xlsx')
countries = pd.read_csv('Countries.csv')

print('Contacts:', contacts.shape, '| columns:', contacts.columns.tolist())
print()
print('Searches:', searches.shape, '| columns:', searches.columns.tolist())

In [ ]:
# ── Column aliases (used throughout notebook) ────────────────────────────────
USER_COL    = 'id_user'
GUEST_COL   = 'id_guest'
COUNTRY_COL = 'origin_country'
PRICE_COL   = 'uses_price_filter'
ROOM_COL    = 'uses_room_filter'
LT_ORDER    = ['Last Minute (0-7d)', 'Short (8-30d)', 'Medium (31-90d)', 'Long (90d+)']

# ── Parse timestamps ─────────────────────────────────────────────────────────
for col in ['ts_contact_at', 'ts_reply_at', 'ts_accepted_at', 'ts_booking_at',
            'ds_checkin', 'ds_checkout']:
    contacts[col] = pd.to_datetime(contacts[col], errors='coerce')

for col in ['ds', 'ds_checkin', 'ds_checkout']:
    searches[col] = pd.to_datetime(searches[col], errors='coerce')

# ── Rebuild lead_time_category from the numeric columns ─────────────────────
#    (avoids relying on Excel text values which may differ)
lt_bins   = [-1, 7, 30, 90, float('inf')]

contacts['lead_time_category'] = pd.cut(
    contacts['lead_time_days'], bins=lt_bins, labels=LT_ORDER
)

searches['lead_time_category'] = pd.cut(
    searches['search_lead_time'], bins=lt_bins, labels=LT_ORDER
)

print('Data prepared.')
print(contacts[['is_replied','is_accepted','is_booked','reply_time_minutes','lead_time_days']].describe().round(1))

## A3. Conversion Funnel – Methodology

The Airbnb conversion funnel has 5 stages:

```
Search → Contact → Reply → Accept → Book
```

Each stage counts **unique users**, not rows. This matters because a single user may perform dozens of searches before sending one contact request.

| Stage | Source | Counted as |
|---|---|---|
| **Search** | `searches` | Unique `id_user` |
| **Contact** | `contacts` | Unique `id_guest` |
| **Reply** | `contacts` | Unique `id_guest` where `is_replied = 1` |
| **Accept** | `contacts` | Unique `id_guest` where `is_accepted = 1` |
| **Book** | `contacts` | Unique `id_guest` where `is_booked = 1` |

In [ ]:
n_search  = searches[USER_COL].nunique()
n_contact = contacts[GUEST_COL].nunique()
n_reply   = contacts.loc[contacts['is_replied']  == 1, GUEST_COL].nunique()
n_accept  = contacts.loc[contacts['is_accepted'] == 1, GUEST_COL].nunique()
n_book    = contacts.loc[contacts['is_booked']   == 1, GUEST_COL].nunique()

funnel = pd.DataFrame({
    'Stage': ['Search', 'Contact', 'Reply', 'Accept', 'Book'],
    'Users': [n_search, n_contact, n_reply, n_accept, n_book]
})
funnel['% of Searches']    = (funnel['Users'] / n_search * 100).round(1)
funnel['Stage-to-Stage %'] = [
    100.0,
    round(n_contact / n_search  * 100, 1),
    round(n_reply   / n_contact * 100, 1),
    round(n_accept  / n_reply   * 100, 1),
    round(n_book    / n_accept  * 100, 1),
]
print(funnel.to_string(index=False))

In [ ]:
colors = [RED, '#FF8A8D', '#FFB3B5', '#FFD4D5', '#FFF0F0']
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(funnel['Stage'][::-1], funnel['Users'][::-1], color=colors)

for bar, (_, row) in zip(bars, funnel[::-1].iterrows()):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f"{row['Users']:,}  ({row['% of Searches']}%)",
            va='center', fontsize=10)

ax.set_xlabel('Unique Users')
ax.set_title('Airbnb Dublin – Conversion Funnel (Unique Users)', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('funnel.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part B – Platform-Wide Findings

## B1. Search Filter Usage – Intent Signal

In [ ]:
pct_price = searches[PRICE_COL].mean() * 100
pct_room  = searches[ROOM_COL].mean()  * 100
print(f'Price filter usage: {pct_price:.1f}%')
print(f'Room type filter usage: {pct_room:.1f}%')

# Tag each search row with whether the user eventually booked
bookers = set(contacts.loc[contacts['is_booked'] == 1, GUEST_COL])
searches['did_book'] = searches[USER_COL].isin(bookers).astype(int)

In [ ]:
# Booking rate by filter combination
filter_groups  = searches.groupby([PRICE_COL, ROOM_COL])[USER_COL].nunique().reset_index(name='searchers')
filter_bookers = (searches[searches['did_book']==1]
                  .groupby([PRICE_COL, ROOM_COL])[USER_COL].nunique().reset_index(name='bookers'))

filter_conv = filter_groups.merge(filter_bookers, on=[PRICE_COL, ROOM_COL], how='left').fillna(0)
filter_conv['booking_rate'] = (filter_conv['bookers'] / filter_conv['searchers'] * 100).round(1)

filter_conv['label'] = filter_conv.apply(
    lambda r: f"Price={'Yes' if r[PRICE_COL]==1 else 'No'}, Room={'Yes' if r[ROOM_COL]==1 else 'No'}", axis=1
)
print(filter_conv[['label','searchers','bookers','booking_rate']].to_string(index=False))

In [ ]:
rates  = filter_conv['booking_rate'].values
labels = filter_conv['label'].values
colors = [RED if r == max(rates) else TEAL for r in rates]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(labels, rates, color=colors, edgecolor='white')
for bar, r in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{r:.1f}%', ha='center', fontsize=10)
ax.set_ylabel('Booking Rate (%)')
ax.set_title('Booking Rate by Search Filter Combination', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('filter_conversion.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding:** Users who apply both price and room-type filters show significantly higher booking rates than those who search without filters. Filter usage is a strong **intent signal** — filtered searches indicate a guest who knows what they want and is close to booking.

## B2. Search Intensity – How Many Searches Before Booking?

In [ ]:
# searches table already has n_searches per user per row — aggregate to user level
user_searches = searches.groupby(USER_COL).agg(
    total_searches=('n_searches', 'sum'),
    did_book=('did_book', 'max')
).reset_index()

bins   = [0, 1, 5, 15, 50, float('inf')]
labels = ['1', '2-5', '6-15', '16-50', '50+']
user_searches['intensity'] = pd.cut(user_searches['total_searches'], bins=bins, labels=labels)

intensity_conv = user_searches.groupby('intensity', observed=True).agg(
    users=('did_book', 'count'),
    bookers=('did_book', 'sum')
).reset_index()
intensity_conv['booking_rate'] = (intensity_conv['bookers'] / intensity_conv['users'] * 100).round(1)
print(intensity_conv.to_string(index=False))

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

x = intensity_conv['intensity'].astype(str)
ax1.bar(x, intensity_conv['users'],        color=TEAL, alpha=0.6, label='Users')
ax2.plot(x, intensity_conv['booking_rate'], color=RED,  marker='o', linewidth=2, label='Booking Rate %')

ax1.set_xlabel('Number of Searches per User')
ax1.set_ylabel('Users', color=TEAL)
ax2.set_ylabel('Booking Rate (%)', color=RED)
ax1.set_title('Search Intensity vs Booking Rate', fontsize=12, fontweight='bold')
lines1, lbl1 = ax1.get_legend_handles_labels()
lines2, lbl2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, lbl1+lbl2, loc='upper left')
plt.tight_layout()
plt.savefig('search_intensity.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding:** Booking rate rises sharply with search intensity. High-intensity users (50+ searches) convert at over 50% — the platform's most valuable audience. This supports targeted re-engagement campaigns for high-intent non-bookers.

## B3. Lead Time – When Do Guests Plan, and Does It Matter?

In [ ]:
lt_rates = contacts.groupby('lead_time_category', observed=True).agg(
    n_contacts=(GUEST_COL, 'count'),
    reply_rate=('is_replied',  'mean'),
    accept_rate=('is_accepted', 'mean'),
    book_rate=('is_booked',   'mean')
).reset_index()
lt_rates[['reply_rate','accept_rate','book_rate']] = (
    lt_rates[['reply_rate','accept_rate','book_rate']] * 100
).round(1)
print(lt_rates.to_string(index=False))

In [ ]:
x     = range(len(lt_rates))
w     = 0.25
xlbls = lt_rates['lead_time_category'].astype(str).tolist()

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar([i-w for i in x], lt_rates['reply_rate'],  w, label='Reply Rate',  color='#8ECFC9')
ax.bar([i   for i in x], lt_rates['accept_rate'], w, label='Accept Rate', color=TEAL)
ax.bar([i+w for i in x], lt_rates['book_rate'],   w, label='Book Rate',   color=RED)

ax.set_xticks(list(x))
ax.set_xticklabels(xlbls, rotation=10)
ax.set_ylabel('Rate (%)')
ax.set_title('Conversion Rates by Lead Time Category', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('lead_time.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding:** Lead time primarily affects the **Accept stage**, not Reply. The 8–30 day window is the platform's "golden window" — hosts accept most readily when guests plan ahead but not too far ahead. Last-minute contacts (0–7 days) and very far-advance contacts (90+ days) both underperform.

## B4. Idle Listings – Supply-Side Friction

In [ ]:
listing_stats = contacts.groupby('id_listing').agg(
    total_contacts=(GUEST_COL,    'count'),
    total_accepts=('is_accepted', 'sum'),
    total_bookings=('is_booked',  'sum')
).reset_index()

idle   = listing_stats[(listing_stats['total_contacts'] > 0) & (listing_stats['total_bookings'] == 0)]
active = listing_stats[listing_stats['total_bookings'] > 0]

print(f'Idle listings  (contacts but 0 bookings): {len(idle):,}')
print(f'Active listings (≥1 booking):              {len(active):,}')
print()

for name, df in [('Idle', idle), ('Active', active)]:
    ar = (df['total_accepts'] / df['total_contacts'].clip(1)).mean() * 100
    print(f'{name}: avg contacts = {df["total_contacts"].mean():.1f}  |  accept rate = {ar:.1f}%')

In [ ]:
idle['accept_rate']   = idle['total_accepts']   / idle['total_contacts'].clip(1) * 100
active['accept_rate'] = active['total_accepts'] / active['total_contacts'].clip(1) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, df, title, color in zip(
    axes,
    [idle, active],
    ['Idle Listings', 'Active Listings'],
    [RED, TEAL]
):
    ax.hist(df['accept_rate'].clip(0, 100), bins=20, color=color, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Accept Rate (%)')
    ax.set_ylabel('Number of Listings')

plt.suptitle('Accept Rate Distribution: Idle vs Active Listings', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('idle_listings.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding:** Idle listings receive guest contacts but never convert — not because guests don't reach out, but because host accept rates are extremely low. These listings represent a major source of funnel leakage: guests exhaust their contacts on non-converting hosts and leave the platform.

**Recommendation:** De-rank or suspend idle listings until host engagement improves. A host health score based on accept rate and response frequency could surface this issue proactively.

---
# Part C – Market-by-Market Analysis

## C1. Volume vs Efficiency – Top 7 Origin Countries

In [ ]:
# Searchers per country
search_by_country = (
    searches.groupby(COUNTRY_COL)[USER_COL].nunique()
    .reset_index(name='searchers')
)

# Bookers per country — origin_country is already in contacts
bookers_by_country = (
    contacts[contacts['is_booked'] == 1]
    .groupby(COUNTRY_COL)[GUEST_COL].nunique()
    .reset_index(name='bookers')
)

market = search_by_country.merge(bookers_by_country, on=COUNTRY_COL, how='left')
market['bookers']          = market['bookers'].fillna(0).astype(int)
market['conversion_rate']  = (market['bookers'] / market['searchers'] * 100).round(1)
market['bookers_per_1000'] = (market['bookers'] / market['searchers'] * 1000).round(1)

top7 = market.nlargest(7, 'searchers')
print(top7[['origin_country','searchers','bookers','conversion_rate','bookers_per_1000']].to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(top7[COUNTRY_COL][::-1], top7['searchers'][::-1], color=TEAL)
ax1.set_xlabel('Unique Searchers')
ax1.set_title('Search Volume by Country', fontweight='bold')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

bar_colors = [RED if c=='IE' else DARK for c in top7[COUNTRY_COL][::-1]]
ax2.barh(top7[COUNTRY_COL][::-1], top7['bookers_per_1000'][::-1], color=bar_colors)
ax2.set_xlabel('Bookers per 1,000 Searchers')
ax2.set_title('Market Efficiency by Country', fontweight='bold')
for bar, val in zip(ax2.patches, top7['bookers_per_1000'][::-1]):
    ax2.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
             f'{val}', va='center', fontsize=9)

plt.suptitle('Volume vs Efficiency – Top 7 Origin Markets', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('market_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## C2. The Ireland Anomaly

Ireland (IE) generates the highest search volume yet has one of the lowest conversion rates. We tested two hypotheses:

In [ ]:
ie_searchers    = set(searches.loc[searches[COUNTRY_COL]=='IE', USER_COL])
all_hosts       = set(contacts['id_host'])
host_overlap    = ie_searchers & all_hosts
pct_hosts       = len(host_overlap) / len(ie_searchers) * 100

print(f'IE unique searchers:             {len(ie_searchers):,}')
print(f'IE searchers who are also hosts: {len(host_overlap):,} ({pct_hosts:.1f}%)')

ie_bookers = contacts[(contacts[COUNTRY_COL]=='IE') & (contacts['is_booked']==1)][GUEST_COL].nunique()
ie_pure    = ie_searchers - all_hosts
ie_bookers_pure = (
    contacts[(contacts[COUNTRY_COL]=='IE') &
             (contacts['is_booked']==1) &
             (~contacts[GUEST_COL].isin(all_hosts))][GUEST_COL].nunique()
)

conv_all  = ie_bookers / len(ie_searchers) * 100
conv_pure = ie_bookers_pure / len(ie_pure) * 100 if ie_pure else 0

print(f'\nIE conversion rate (all searchers):             {conv_all:.1f}%')
print(f'IE conversion rate (excl. host-searchers):      {conv_pure:.1f}%')
print('\nConclusion: Host overlap is a partial but not the primary explanation.')

**Hypothesis 2 – Lower contact-to-booking ratio:** Even after removing host-searchers from the IE pool, conversion only improves modestly. The deeper driver is that IE guests have a significantly lower contact-to-booking conversion compared to markets like the US and France — possibly reflecting local habits (using Airbnb for price discovery) or competition from local rental platforms.

## C3. The France Model – High Efficiency Through Filter Usage

In [ ]:
market_filter = []
for country in ['FR', 'US', 'IE', 'GB']:
    sub = searches[searches[COUNTRY_COL] == country]
    if len(sub) == 0:
        continue
    market_filter.append({
        'Country': country,
        'N searches': len(sub),
        'Price filter %': round(sub[PRICE_COL].mean() * 100, 1),
        'Room filter %':  round(sub[ROOM_COL].mean()  * 100, 1),
        'Conversion %':   round(
            market.set_index(COUNTRY_COL).loc[country, 'conversion_rate']
            if country in market[COUNTRY_COL].values else 0, 1
        )
    })

mf = pd.DataFrame(market_filter)
print(mf.to_string(index=False))

In [ ]:
x = range(len(mf))
w = 0.3

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([i-w/2 for i in x], mf['Price filter %'], w, label='Price Filter %', color=TEAL)
ax.bar([i+w/2 for i in x], mf['Room filter %'],  w, label='Room Filter %',  color='#8ECFC9')

ax2 = ax.twinx()
ax2.plot(list(x), mf['Conversion %'], color=RED, marker='D', linewidth=2, label='Conversion %')
ax2.set_ylabel('Conversion Rate (%)', color=RED)

ax.set_xticks(list(x))
ax.set_xticklabels(mf['Country'])
ax.set_ylabel('Filter Usage (%)')
ax.set_title('Filter Usage vs Conversion Rate by Market', fontsize=12, fontweight='bold')
lines1, l1 = ax.get_legend_handles_labels()
lines2, l2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, l1+l2, loc='upper left')
plt.tight_layout()
plt.savefig('france_model.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding:** France has the highest filter usage rate and one of the highest conversion rates. Filter usage and conversion move together across markets — France is the benchmark for what high-intent search behavior looks like at scale.

## C4. Timezone Effect – US vs Europe

Airbnb Dublin hosts are in Ireland (GMT+1 in summer). US guests (EST = GMT-5) contact hosts during their afternoon — which is the middle of the Dublin night.

In [ ]:
contacts['contact_hour_utc']    = contacts['ts_contact_at'].dt.hour
contacts['contact_hour_dublin'] = (contacts['contact_hour_utc'] + 1) % 24   # UTC → Dublin (GMT+1)

for market in ['US', 'IE', 'FR', 'GB']:
    sub = contacts[contacts[COUNTRY_COL] == market]
    if len(sub) == 0:
        continue
    off_pct    = ((sub['contact_hour_dublin'] >= 22) | (sub['contact_hour_dublin'] < 8)).mean() * 100
    median_rep = sub['reply_time_minutes'].median()
    print(f"{market}: off-hours contacts = {off_pct:.1f}%  |  median reply time = {median_rep:.0f} min")

In [ ]:
us_hours = contacts[contacts[COUNTRY_COL]=='US']['contact_hour_dublin'].dropna()
ie_hours = contacts[contacts[COUNTRY_COL]=='IE']['contact_hour_dublin'].dropna()

fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(ie_hours, bins=24, range=(0,24), alpha=0.6, label='Ireland (IE)', color=TEAL, density=True)
ax.hist(us_hours, bins=24, range=(0,24), alpha=0.6, label='United States (US)', color=RED, density=True)
ax.axvspan(0, 8, alpha=0.07, color='gray', label='Dublin sleep hours (00–08)')
ax.axvspan(22, 24, alpha=0.07, color='gray')
ax.set_xlabel('Hour of Day (Dublin Local Time)')
ax.set_ylabel('Density')
ax.set_title('When Do Guests Contact Hosts? (Dublin Local Time)', fontsize=12, fontweight='bold')
ax.set_xticks(range(0, 25, 2))
ax.legend()
plt.tight_layout()
plt.savefig('timezone_contact.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding:** A significant share of US contacts arrive during Dublin's nighttime hours, leading to longer reply times. Longer reply times are associated with lower booking completion.

**Recommendation:** Show US-based guests a timezone-aware message when contacting Dublin hosts (e.g. *"Your host is in a different timezone — expect a reply by morning Dublin time"*). This reduces abandonment without requiring host behavior change.

UK guests (GMT+0/+1) have near-zero timezone gap with Dublin and represent a strong organic growth opportunity.

---
# Summary – Business Recommendations

| # | Finding | Recommendation | Expected Impact |
|---|---|---|---|
| 1 | Filter users book at 3× higher rates | Prompt guests to use filters before browsing — nudge on empty search | ↑ Contact quality, ↑ Booking rate |
| 2 | High-intensity searchers (50+) convert at 50%+ | Trigger targeted push / email for high-intent non-bookers | ↑ Booking volume |
| 3 | 8–30 day lead time = acceptance "golden window" | Remind guests to book ahead; warn last-minute searchers of lower odds | ↑ Accept rate |
| 4 | Idle listings waste guest contacts | Host health score; de-rank listings with accept rate < 30% | ↓ Guest frustration, ↑ Funnel efficiency |
| 5 | US contacts arrive during Dublin night hours | Timezone-aware reply-time message for cross-timezone contacts | ↓ US abandonment |
| 6 | IE: high volume, low conversion | Investigate local alternatives; consider IE-specific promotions | ↑ IE conversion |
| 7 | FR converts best via filter usage | Use FR as UX benchmark for search onboarding in other markets | ↑ Platform-wide efficiency |